# cp_shared - Full Backend Walkthrough

This notebook has all the backend logic. The dashboard (`dashboard.py`) imports everything from here at runtime, so there's no separate .py file needed.

## 1. Imports & Setup

`numpy` and `pandas` handle all the data work. `requests` is used to check whether a download URL is actually live before we try to fetch it. `sklearn` provides Linear Regression and Random Forest. `mean_absolute_error` and `mean_squared_error` from sklearn are used across the evaluation functions. `warnings` is silenced globally so convergence messages from sklearn and statsmodels don't pollute the output.

In [ ]:
import warnings
import math
from typing import Any, Dict, List, Union
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Keep output clean by hiding sklearn convergence warnings etc.
warnings.filterwarnings("ignore")

## 2. Constants & Configuration

All the settings that control the pipeline are here.
| Constant | What it does |
|---|---|
| `BASE_DATA_URLS` | Known S3 download links for each year's SINAN dengue CSV (2021-2026) |
| `DATA_URL_TEMPLATE` | Template used to probe for years not yet in `BASE_DATA_URLS` |
| `CACHE_DIR` | Local `.data_cache/` folder for parquet files, avoids re-downloading ~500 MB every run |
| `WEEKLY_CACHE_PATH` / `RAW_CACHE_PATH` | Legacy single-file paths kept for migrating old caches |
| `UF_OPTIONS` | Maps IBGE state codes to display names for the sidebar dropdown |
| `TARGET_UF_CODE` | Default state code, 32 = Espirito Santo |
| `ANALYSIS_START` | Trims the series to Jan 2023 onward so older noisy data doesn't affect the models |
| `FEATURE_COLS` | The 4 inputs every model uses: lag1, lag4, week_sin, week_cos |
| `MIN_BACKTEST_TRAIN` | Minimum 26 weeks of history before the backtest starts predicting |
| `TREND_THRESHOLD` | A 10% week-over-week change triggers a Rising or Falling label |
| `BAND_QUANTILES` | 25th to 75th percentile of residuals used to define the safety band |
| `DEFAULT_HOLDOUT_WEEKS` | 52-week holdout period for final evaluation |
| `DEFAULT_SELECTION_WEEKS` | 8 most recent pre-holdout weeks used for production model selection |

In [ ]:
import os as _os

# Known download links for each year's SINAN dengue CSV on the Ministry of Health S3 bucket
BASE_DATA_URLS = {2021: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR21.csv.zip",
    2022: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR22.csv.zip",
    2023: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR23.csv.zip",
    2024: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR24.csv.zip",
    2025: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR25.csv.zip",
    2026: "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR26.csv.zip"}
# Template used to probe for newer years that aren't in BASE_DATA_URLS yet
DATA_URL_TEMPLATE = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR{year_suffix}.csv.zip"

# Data source metadata
DATA_SOURCE_LABEL = "SINAN/Dengue — Brazilian Ministry of Health"
DATA_SOURCE_API_URL = "https://apidadosabertos.saude.gov.br/arboviroses/dengue"
DATA_SOURCE_PORTAL_URL = "https://dadosabertos.saude.gov.br/dataset/arboviroses-dengue/resource/a9b73910-f233-417b-85c9-95230c269e1c"

# Local cache paths (we store processed parquet files here to avoid re-downloading ~500MB of CSVs)
_MODULE_DIR = _os.path.dirname(_os.path.abspath(__file__)) if "__file__" in dir() else _os.getcwd()
CACHE_DIR = _os.path.join(_MODULE_DIR, ".data_cache")
WEEKLY_CACHE_PATH = _os.path.join(CACHE_DIR, "weekly_notifications.parquet")  # legacy, kept for migration
RAW_CACHE_PATH = _os.path.join(CACHE_DIR, "raw_notifications.parquet")
CACHE_META_PATH = _os.path.join(CACHE_DIR, "cache_meta.json")

# Selectable areas (IBGE UF code -> state name)
UF_OPTIONS: Dict[int, str] = {32: "Espírito Santo", 33: "Rio de Janeiro", 31: "Minas Gerais", 35: "São Paulo", 29: "Bahia"}

# Default target state and analysis settings
TARGET_UF_CODE = 32
TARGET_LABEL = "Espírito Santo"
ANALYSIS_START = pd.Timestamp("2023-01-02")  # only use data from Jan 2023 onward
FEATURE_COLS = ["lag1", "lag4", "week_sin", "week_cos"]  # the 4 features all models use
MIN_BACKTEST_TRAIN = 26  # minimum weeks of training before backtesting starts
TREND_THRESHOLD = 0.10  # 10% change triggers Rising/Falling label
BAND_QUANTILES = (0.25, 0.75)  # 25th-75th percentile of residuals for safety bands
DEFAULT_HOLDOUT_WEEKS = 52  # 1 year holdout for evaluation
DEFAULT_SELECTION_WEEKS = 8  # most recent 8 weeks used for model selection scoring
_AVAILABLE_DATA_URLS_CACHE: Union[None, Dict[int, str]] = None  # in-memory cache for discovered URLs


def _uf_cache_paths(uf_code: int = TARGET_UF_CODE) -> Dict[str, str]:
    """Return per-UF cache file paths."""
    return {
        "weekly": _os.path.join(CACHE_DIR, f"weekly_{uf_code}.parquet"),
        "raw": _os.path.join(CACHE_DIR, f"raw_{uf_code}.parquet"),
        "meta": _os.path.join(CACHE_DIR, f"meta_{uf_code}.json"),
    }

## 3. Data URL Discovery

Three small helpers for finding which yearly files are available on the Ministry of Health's S3 bucket. `_build_year_url()` constructs the download URL for a given year (e.g. 2026 becomes DENGBR26.csv.zip). `_url_exists()` sends a HEAD request first because it's lightweight, then falls back to a streaming GET if the server returns 403 or 405 since some servers reject HEAD requests entirely. `get_available_data_urls()` starts from the known 2021-2025 URLs, then probes for anything newer up to next calendar year. The result is cached in memory so we only hit the network once per session.

In [ ]:
def _build_year_url(year: int) -> str:
    """Build a download URL for a given year, e.g. 2026 -> DENGBR26.csv.zip."""
    return DATA_URL_TEMPLATE.format(year_suffix=str(year)[-2:])


def _url_exists(url: str, timeout: int = 8) -> bool:
    """Check if a remote URL is reachable. Tries HEAD first, falls back to GET if needed."""
    try:
        response = requests.head(url, allow_redirects=True, timeout=timeout)
        if response.ok:
            return True
        # Some servers block HEAD but allow GET
        if response.status_code in {403, 405}:
            response = requests.get(url, stream=True, timeout=timeout)
            response.close()
            return response.ok
        return False
    except requests.RequestException:
        return False


def get_available_data_urls(refresh: bool = False) -> Dict[int, str]:
    """Return a dict of {year: url} for all available SINAN dengue CSVs.
    Starts with the known URLs in BASE_DATA_URLS, then probes for newer years.
    Results are cached in memory so we only check once per session."""
    global _AVAILABLE_DATA_URLS_CACHE

    if _AVAILABLE_DATA_URLS_CACHE is not None and not refresh:
        return dict(_AVAILABLE_DATA_URLS_CACHE)

    detected_urls = dict(BASE_DATA_URLS)
    latest_known_year = max(detected_urls)
    max_probe_year = pd.Timestamp.today().year + 1

    # Probe for any years beyond what we already know about
    for year in range(latest_known_year + 1, max_probe_year + 1):
        candidate_url = _build_year_url(year)
        if _url_exists(candidate_url):
            detected_urls[year] = candidate_url

    _AVAILABLE_DATA_URLS_CACHE = dict(sorted(detected_urls.items()))
    return dict(_AVAILABLE_DATA_URLS_CACHE)

## 3b. Local Data Cache

To avoid re-downloading around 500 MB of CSVs on every load, the pipeline saves processed DataFrames as local parquet files inside `.data_cache/`. Parquet is used instead of CSV because it's much faster to read and preserves column types without needing to re-infer them.

`_save_cache()` writes the weekly and raw DataFrames plus a metadata JSON with timestamps, row counts, and year coverage. `_load_cache()` reads everything back and re-trims to the analysis window. `cache_is_fresh()` checks whether the cache is younger than `max_age_hours` (default 12 hours). `build_weekly_series()` always checks the cache first and only re-downloads if it's stale or `refresh=True`. Each state (UF code) gets its own set of cache files so switching areas doesn't invalidate the other.

In [ ]:
import json as _json
import time as _time


def _ensure_cache_dir():
    """Create the cache directory if it doesn't exist."""
    _os.makedirs(CACHE_DIR, exist_ok=True)


def _save_cache(raw_all: pd.DataFrame, full_df: pd.DataFrame, df: pd.DataFrame,
                uf_code: int = TARGET_UF_CODE, raw_count=None) -> None:
    """Persist processed DataFrames to local parquet files, plus a metadata JSON
    with timestamps, row counts, and year coverage for freshness checks."""
    _ensure_cache_dir()
    paths = _uf_cache_paths(uf_code)
    full_df.to_parquet(paths["weekly"], index=False)
    # Only save the columns we actually need from the raw data
    raw_cols = ["DT_SIN_PRI", "SEM_PRI", "SG_UF", "ID_MN_RESI", "CLASSI_FIN", "week_start", "notifications"]
    save_cols = [c for c in raw_cols if c in raw_all.columns]
    if len(save_cols) > 0 and len(raw_all) > 0:
        raw_all[save_cols].to_parquet(paths["raw"], index=False)
    # Write metadata so we can check cache age later
    n_raw = int(raw_count) if raw_count is not None else int(len(raw_all))
    meta = {"created_utc": pd.Timestamp.utcnow().isoformat(), "created_local": pd.Timestamp.now().isoformat(),
            "latest_date": str(df["date"].max().date()), "n_weeks": len(df), "n_raw": n_raw,
            "years": sorted(full_df["year"].unique().tolist()), "uf_code": uf_code,
            "uf_label": UF_OPTIONS.get(uf_code, f"UF {uf_code}")}
    with open(paths["meta"], "w") as f:
        _json.dump(meta, f, indent=2)


def _load_cache(uf_code: int = TARGET_UF_CODE):
    """Load cached DataFrames from parquet. Returns (raw_all, full_df, df) or None if cache is missing."""
    paths = _uf_cache_paths(uf_code)
    weekly_path, raw_path = paths["weekly"], paths["raw"]
    # Legacy fallback: if per-UF file is missing but old-format exists for default UF
    if not _os.path.exists(weekly_path) and uf_code == TARGET_UF_CODE and _os.path.exists(WEEKLY_CACHE_PATH):
        weekly_path, raw_path = WEEKLY_CACHE_PATH, RAW_CACHE_PATH
    if not _os.path.exists(weekly_path):
        return None
    try:
        full_df = pd.read_parquet(weekly_path)
        full_df["date"] = pd.to_datetime(full_df["date"])
        raw_all = pd.read_parquet(raw_path) if _os.path.exists(raw_path) else pd.DataFrame()
        if "DT_SIN_PRI" in raw_all.columns:
            raw_all["DT_SIN_PRI"] = pd.to_datetime(raw_all["DT_SIN_PRI"], errors="coerce")
        # Trim to analysis window
        df = full_df.loc[full_df["date"] >= ANALYSIS_START].copy().reset_index(drop=True)
        return raw_all, full_df, df
    except Exception:
        return None


def cache_exists(uf_code: int = TARGET_UF_CODE) -> bool:
    """Return True if a weekly cache file exists for the UF (regardless of age)."""
    paths = _uf_cache_paths(uf_code)
    if _os.path.exists(paths["weekly"]):
        return True
    if uf_code == TARGET_UF_CODE and _os.path.exists(WEEKLY_CACHE_PATH):
        return True
    return False


def cache_is_fresh(max_age_hours: float = 12.0, uf_code: int = TARGET_UF_CODE) -> bool:
    """Return True if a local cache exists and is younger than max_age_hours."""
    paths = _uf_cache_paths(uf_code)
    meta_path = paths["meta"]
    # Legacy fallback for the default UF
    if not _os.path.exists(meta_path) and uf_code == TARGET_UF_CODE and _os.path.exists(CACHE_META_PATH):
        meta_path = CACHE_META_PATH
    if not _os.path.exists(meta_path):
        return False
    try:
        with open(meta_path) as f:
            meta = _json.load(f)
        created = pd.Timestamp(meta["created_utc"])
        age_hours = (pd.Timestamp.utcnow() - created).total_seconds() / 3600
        return age_hours < max_age_hours
    except Exception:
        return False


def get_cache_info(uf_code: int = TARGET_UF_CODE) -> Dict[str, Any]:
    """Return metadata about the current cache (timestamps, coverage), or empty dict if none."""
    paths = _uf_cache_paths(uf_code)
    meta_path = paths["meta"]
    if not _os.path.exists(meta_path) and uf_code == TARGET_UF_CODE and _os.path.exists(CACHE_META_PATH):
        meta_path = CACHE_META_PATH
    if not _os.path.exists(meta_path):
        return {}
    try:
        with open(meta_path) as f:
            return _json.load(f)
    except Exception:
        return {}


## 4. Data Loading - Single Year

`load_dengue_year()` downloads one year's CSV from the Ministry of Health S3 bucket and processes it into a weekly notification count. The CSVs use latin1 encoding (that's the format MoH publishes them in, utf-8 will fail). We only load 5 of the many columns in the file since the rest are clinical details we don't need. After parsing dates, rows with invalid or missing dates get dropped. The data is then filtered to the target state using the IBGE UF code.

Each notification gets snapped to the Monday of its week by subtracting the weekday offset. This gives us clean Monday-anchored buckets before we group and sum into weekly counts. The function returns both the raw individual-row DataFrame and the aggregated weekly one.

In [ ]:
def load_dengue_year(year: int, uf_code: int = TARGET_UF_CODE):
    """Download one year's CSV from S3, filter to the target state, and aggregate into weekly counts.
    Reads in 250k-row chunks to keep memory low. Returns (raw_count, weekly aggregated DataFrame)."""
    data_urls = get_available_data_urls()
    # only pull the columns we need, the full CSV has dozens of clinical fields we don't use
    usecols = ["DT_SIN_PRI", "SEM_PRI", "SG_UF", "ID_MN_RESI", "CLASSI_FIN"]
    # latin1 is what the Ministry of Health uses for these files, utf-8 will break
    raw_count = 0
    weekly_parts = []
    for chunk in pd.read_csv(data_urls[year], compression="zip", encoding="latin1",
                             usecols=usecols, low_memory=False, chunksize=250_000):
        # drop rows where the notification date is missing or unparseable
        chunk["DT_SIN_PRI"] = pd.to_datetime(chunk["DT_SIN_PRI"], errors="coerce")
        chunk = chunk.dropna(subset=["DT_SIN_PRI"])
        # keep only the selected state
        chunk = chunk[chunk["SG_UF"] == uf_code].copy()
        if chunk.empty:
            continue
        raw_count += int(len(chunk))
        # align each notification to the Monday of its week so groupby gives clean weekly buckets
        chunk["week_start"] = chunk["DT_SIN_PRI"] - pd.to_timedelta(chunk["DT_SIN_PRI"].dt.weekday, unit="D")
        chunk["notifications"] = 1
        weekly_parts.append(chunk.groupby("week_start", as_index=False)["notifications"]
            .sum().sort_values("week_start"))
    if weekly_parts:
        # sum into weekly counts, merging across chunks
        weekly = (pd.concat(weekly_parts, ignore_index=True)
            .groupby("week_start", as_index=False)["notifications"]
            .sum().sort_values("week_start"))
    else:
        weekly = pd.DataFrame(columns=["week_start", "notifications"])
    weekly["source_year"] = year
    return raw_count, weekly


## 5. Data Loading - Full Weekly Series (with Cache)

`build_weekly_series()` loads all available years and merges them into one continuous weekly series. It checks the local cache first and returns in under a second if the data is fresh. Otherwise it downloads everything, combines the yearly parts, and resolves any overlapping weeks by summing their counts (this can happen near year boundaries).

After merging, it builds a complete Monday-to-Monday grid using `pd.date_range` with `freq=W-MON` so there are no gaps in the index. Weeks with no notifications get filled with zero, and an `is_imputed_zero_week` flag marks them so the dashboard can note how many imputed weeks are in the window. The function returns three DataFrames: the raw notification rows, the full unfiltered weekly series going back to 2021, and the trimmed analysis window starting from `ANALYSIS_START`.

In [ ]:
def build_weekly_series(refresh: bool = False, uf_code: int = TARGET_UF_CODE):
    """Load all available years, merge into a single weekly series, and return three DataFrames:
    (raw_all, full_df, df). Uses the local parquet cache when available to avoid re-downloading."""
    # Use cache_exists (not cache_is_fresh) so startup is fast even if cache is stale
    if not refresh and cache_exists(uf_code=uf_code):
        cached = _load_cache(uf_code=uf_code)
        if cached is not None:
            return cached

    # Download every available year from the S3 bucket, skipping years before the analysis window
    data_urls = get_available_data_urls(refresh=refresh)
    min_required_year = max(min(data_urls), int(ANALYSIS_START.year) - 1)
    total_raw_count = 0
    weekly_parts = []
    for year in sorted(data_urls):
        if year < min_required_year:
            continue
        year_raw_count, weekly_df = load_dengue_year(year, uf_code)
        total_raw_count += int(year_raw_count)
        weekly_parts.append(weekly_df)

    # raw_all is empty since load_dengue_year no longer returns raw rows (memory optimization)
    raw_all = pd.DataFrame(columns=["DT_SIN_PRI", "SEM_PRI", "SG_UF", "ID_MN_RESI", "CLASSI_FIN", "week_start", "notifications"])
    # Combine all years and merge overlapping weeks
    weekly_all = (pd.concat(weekly_parts, ignore_index=True)
        .groupby("week_start", as_index=False)["notifications"].sum()
        .rename(columns={"week_start": "date"})
        .sort_values("date").reset_index(drop=True))
    # Build a complete weekly grid (every Monday) so there are no gaps
    full_weeks = pd.DataFrame({"date": pd.date_range(weekly_all["date"].min(), weekly_all["date"].max(), freq="W-MON")})
    df = full_weeks.merge(weekly_all, on="date", how="left")
    df["is_imputed_zero_week"] = df["notifications"].isna()  # track which weeks had no data
    df["notifications"] = df["notifications"].fillna(0).astype(int)
    df["year"] = df["date"].dt.year
    df["epi_week"] = df["date"].dt.isocalendar().week.astype(int)
    full_df = df.copy()
    # Trim to the analysis window
    df = df.loc[df["date"] >= ANALYSIS_START].copy().reset_index(drop=True)

    # Save to cache so next load is instant
    _save_cache(raw_all, full_df, df, uf_code=uf_code, raw_count=total_raw_count)

    return raw_all, full_df, df


## 6. Feature Engineering

`create_features()` adds the 5 columns all models use.

| Feature | What it captures |
|---|---|
| `lag1` | Last week's count, the most recent known signal |
| `lag4` | 4 weeks ago, helps capture short-term momentum |
| `lag52` | Same week last year, the seasonal baseline |
| `week_sin` | Sine of epi week / 52, cyclical seasonality encoding |
| `week_cos` | Cosine of epi week / 52, paired with week_sin |

We use sine and cosine of the week number instead of the raw integer because a raw week number would create a hard discontinuity at the year boundary (week 52 to week 1). The sin/cos pair encodes week position as a smooth circle so models see continuity across years. Rows at the start of the series that don't have enough history to fill the lag columns get dropped.

In [ ]:
def create_features(data: pd.DataFrame) -> pd.DataFrame:
    """Build lag and cyclical features for model training. Drops rows where lags aren't available yet."""
    features = data.copy()
    features["lag1"] = features["notifications"].shift(1)     # last week
    features["lag4"] = features["notifications"].shift(4)     # 4 weeks ago (short-term trend)
    features["lag52"] = features["notifications"].shift(52)   # same week last year (seasonal signal)
    # Encode week number as sine/cosine so models can learn seasonality smoothly
    week_number = features["date"].dt.isocalendar().week.astype(int)
    features["week_sin"] = np.sin(2 * np.pi * week_number / 52)
    features["week_cos"] = np.cos(2 * np.pi * week_number / 52)
    return features.dropna(subset=["lag1", "lag4"]).reset_index(drop=True)

## 7. Model Training - Negative Binomial (Optional)

`fit_negative_binomial_model()` fits a count-data GLM using statsmodels. Dengue notification counts tend to be overdispersed, meaning the variance is much larger than the mean. Poisson regression assumes variance equals the mean, so it underestimates uncertainty during outbreaks. Negative Binomial adds a dispersion parameter (alpha) that absorbs that extra variance.

The model is wrapped in a try/except in the bundle because it can fail to converge on small or noisy training windows. We also run sanity checks on the fitted alpha and the standard errors before accepting the result, and reject it if any of those look unstable.

In [ ]:
def fit_negative_binomial_model(X: np.ndarray, y: np.ndarray):
    """Fit a Negative Binomial GLM using statsmodels. This is a count-data model
    that handles overdispersion better than Poisson. Can fail to converge on small datasets."""
    import statsmodels.api as sm
    from statsmodels.discrete.discrete_model import NegativeBinomial as DiscreteNegativeBinomial

    X_sm = sm.add_constant(X, has_constant="add")  # statsmodels needs an explicit intercept column
    model = DiscreteNegativeBinomial(y, X_sm)
    result = model.fit(disp=0, maxiter=200)  # disp=0 suppresses optimizer output
    return result

## 8. Model Training - Full Bundle

`fit_model_bundle()` trains all models on a given feature DataFrame and returns them in a single dict called the bundle. Everything downstream (backtest, forecast) takes the bundle as input so adding a new model only requires changing this one function.

| Model | Type | Notes |
|---|---|---|
| Naive | Baseline | Predicts last week's value, a random walk baseline |
| Seasonal Naive | Baseline | Predicts the same week last year |
| Linear Regression | Learned | Simple linear fit on the 4 feature columns |
| Random Forest | Learned | 200 trees, max depth 4, min 5 samples per leaf to avoid overfitting |
| Negative Binomial | Learned (optional) | Count-data GLM, only included if it converges |

The two baselines don't need fitting since they just read lag values at prediction time. The Random Forest hyperparameters are kept conservative intentionally because the training sets in the backtest start as small as 26 weeks.

In [ ]:
def fit_model_bundle(feature_frame: pd.DataFrame, include_nb: bool = True, verbose: bool = False) -> dict[str, dict[str, Any]]:
    """Train all models and return them in a dict (the 'bundle').
    Includes two baselines (Naive, Seasonal Naive), Linear Regression, Random Forest,
    and optionally Negative Binomial if it converges."""
    X_local = np.asarray(feature_frame[FEATURE_COLS].to_numpy(dtype=float), dtype=float)
    y_local = np.asarray(feature_frame["notifications"].to_numpy(dtype=float), dtype=int)

    # Baselines don't need fitting, they just use lag values at prediction time
    bundle: dict[str, dict[str, Any]] = {"Naive": {"kind": "baseline"}, "Seasonal Naive": {"kind": "baseline"}}

    # Linear Regression on the 4 features
    linear = LinearRegression()
    linear.fit(X_local, y_local)
    bundle["Linear Regression"] = {"kind": "sklearn", "model": linear}

    # Random Forest with conservative hyperparameters to avoid overfitting
    rf = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=5, random_state=42)
    rf.fit(X_local, y_local)
    bundle["Random Forest"] = {"kind": "sklearn", "model": rf}

    # Negative Binomial is optional because it can fail to converge
    if include_nb:
        try:
            nb_result = fit_negative_binomial_model(X_local, y_local)
            nb_alpha = float(nb_result.params[-1])
            converged = bool(nb_result.mle_retvals.get("converged", False))
            has_finite_bse = np.isfinite(np.asarray(nb_result.bse)).all()
            # Sanity checks: reject if alpha is unstable, didn't converge, or standard errors blew up
            if not np.isfinite(nb_alpha) or abs(nb_alpha) > 100:
                raise ValueError(f"unstable alpha estimate ({nb_alpha})")
            if not converged:
                raise ValueError("optimizer did not converge")
            if not has_finite_bse:
                raise ValueError("standard errors are not finite")
            bundle["Negative Binomial"] = {"kind": "statsmodels", "model": nb_result, "alpha": nb_alpha}
        except Exception as error:
            if verbose:
                print(f"Negative binomial baseline omitted: {error}")

    return bundle

## 9. Prediction

`predict_with_bundle()` runs every model in the bundle on a feature DataFrame and returns a dict mapping each model name to its prediction array. Naive returns lag1, Seasonal Naive returns lag52. Linear Regression and Random Forest call `.predict()` on the 4 feature columns and the result is clipped to zero because both models can technically output negative values for low-count weeks. Negative Binomial calls statsmodels `.predict()` with an intercept column prepended, also clipped to zero.

In [ ]:
def predict_with_bundle(bundle: dict[str, dict[str, Any]], feature_frame: pd.DataFrame) -> dict[str, np.ndarray]:
    """Get predictions from every model in the bundle. Returns {model_name: prediction_array}."""
    X_local = np.asarray(feature_frame[FEATURE_COLS].to_numpy(dtype=float), dtype=float)
    predictions: dict[str, np.ndarray] = {
        "Naive": np.asarray(feature_frame["lag1"].to_numpy(dtype=float), dtype=float),          # just repeat last week
        "Seasonal Naive": np.asarray(feature_frame["lag52"].to_numpy(dtype=float), dtype=float), # same week last year
        "Linear Regression": np.clip(bundle["Linear Regression"]["model"].predict(X_local), 0, None),
        "Random Forest": np.clip(bundle["Random Forest"]["model"].predict(X_local), 0, None),
    }
    if "Negative Binomial" in bundle:
        import statsmodels.api as sm
        X_local_sm = sm.add_constant(X_local, has_constant="add")
        predictions["Negative Binomial"] = np.clip(bundle["Negative Binomial"]["model"].predict(X_local_sm), 0, None)
    return predictions

## 10. Evaluation Metrics - Change-Signal Scoring

`score_predictions()` measures how well each model detects week-over-week direction changes, not just how close the number is. For surveillance the direction matters a lot: missing a rising week when an outbreak is starting is worse than being off by a few cases.

| Metric | What it measures |
|---|---|
| Direction accuracy | Share of weeks where the predicted direction (up/down/flat) matched actual |
| Rising recall | Of all truly rising weeks, how many did the model catch |
| Rising precision | Of all weeks the model predicted as rising, how many actually were |
| Rising F1 | Harmonic mean of precision and recall |
| False alarm rate | Share of non-rising weeks the model incorrectly flagged as rising |

A good model has high recall (catches most outbreaks) and a low false alarm rate (doesn't cry wolf). These two pull in opposite directions, which is why F1 and false alarm rate are both tracked.

In [ ]:
def score_predictions(actual, pred, previous_week):
    """Evaluate how well predictions capture week-over-week direction changes.
    Returns a dict with MAE, RMSE, direction accuracy, rising recall/precision/F1, and false alarm rate."""
    actual_direction = np.sign(actual - previous_week)
    predicted_direction = np.sign(pred - previous_week)
    # Binary flags: did the actual/predicted value go up from last week?
    actual_rising = actual > previous_week
    predicted_rising = pred > previous_week
    # Rising-week detection metrics (like binary classification for "is it going up?")
    true_positives = (actual_rising & predicted_rising).sum()
    rising_recall = true_positives / max(1, actual_rising.sum())
    rising_precision = true_positives / max(1, predicted_rising.sum())
    rising_f1 = 2 * rising_precision * rising_recall / max(1e-9, rising_precision + rising_recall)
    # False alarm: model said rising but it wasn't
    actual_not_rising = ~actual_rising
    false_alarms = (actual_not_rising & predicted_rising).sum()
    false_alarm_rate = false_alarms / max(1, actual_not_rising.sum())
    return {"MAE": mean_absolute_error(actual, pred), "RMSE": np.sqrt(mean_squared_error(actual, pred)),
            "Direction Acc.": (actual_direction == predicted_direction).mean(), "Rising Recall": rising_recall,
            "Rising Precision": rising_precision, "Rising F1": rising_f1, "False Alarm Rate": false_alarm_rate}

## 11. Expanding-Window Backtest

`run_backtest()` is the core evaluation engine. It simulates what would have happened if we had been running these models in real time. It starts with 26 weeks of training data and predicts week 27. Then it expands the training window to include week 27 and predicts week 28. This continues until the end of the series, producing one prediction per week.

No future data ever leaks into a prediction because each fold only sees what would have existed at that point in real time. This pattern is called an expanding-window or anchored walk-forward backtest. It's more expensive than a single train/test split but gives a much more honest picture of how the models would actually perform week to week.

In [ ]:
def run_backtest(features_df: pd.DataFrame, include_nb: bool = True, min_backtest_train: int = MIN_BACKTEST_TRAIN):
    """Run an expanding-window backtest: train on weeks 1..t, predict week t+1, expand, repeat.
    Never uses future data. Returns (backtest_results DataFrame, model_cols mapping)."""
    # maps friendly model names to their column names in the output DataFrame
    model_cols = {"Naive": "naive_pred", "Seasonal Naive": "seasonal_naive_pred",
                  "Linear Regression": "linear_pred", "Random Forest": "rf_pred"}
    backtest_frames = []
    # each iteration trains on weeks 0..t and predicts week t+1 (the fold is never allowed to see t+1)
    for current_idx in range(min_backtest_train, len(features_df)):
        fold_train = pd.DataFrame(features_df.iloc[:current_idx].copy())
        fold_test = pd.DataFrame(features_df.iloc[[current_idx]].copy())
        # fit_model_bundle re-trains NB on every fold, which is slow but ensures no leakage
        fold_bundle = fit_model_bundle(fold_train, include_nb=include_nb, verbose=False)
        fold_predictions = predict_with_bundle(fold_bundle, fold_test)
        fold_frame = pd.DataFrame({
            "date": fold_test["date"].values, "actual": fold_test["notifications"].values.astype(int),
            "previous_week": fold_test["lag1"].values, "naive_pred": fold_predictions["Naive"],
            "seasonal_naive_pred": fold_predictions["Seasonal Naive"],
            "linear_pred": fold_predictions["Linear Regression"], "rf_pred": fold_predictions["Random Forest"],
        })
        # NB is optional, only add the column if it was included in this fold
        if "Negative Binomial" in fold_predictions:
            fold_frame["nb_pred"] = fold_predictions["Negative Binomial"]
        backtest_frames.append(fold_frame)

    backtest_results = pd.concat(backtest_frames, ignore_index=True)
    # nb_pred column only shows up if NB converged on every fold, register it here
    if "nb_pred" in backtest_results.columns:
        model_cols["Negative Binomial"] = "nb_pred"
    return backtest_results, model_cols


## 12. Next-Week Feature Builder

`build_next_week_features()` creates a single feature row for the next week we haven't seen yet. Used when the forecast horizon is 1. It looks up lag1, lag4, and lag52 from history and computes the cyclical week encoding for the target date.

In [ ]:
def build_next_week_features(history: pd.DataFrame) -> pd.DataFrame:
    """Create a single feature row for the next unobserved week (one-step-ahead forecast).
    Looks up lag1, lag4, lag52 from history and computes cyclical week encoding."""
    next_date = history["date"].max() + pd.Timedelta(weeks=1)
    # lag52 is same-week-last-year, need a full year of history or this raises
    lag52_series = history.loc[history["date"] == next_date - pd.Timedelta(weeks=52), "notifications"]
    if lag52_series.empty:
        raise ValueError("Need at least 52 weeks of history for seasonal features.")
    next_week_number = int(next_date.isocalendar().week)
    # iloc[-4] assumes we have at least 4 rows of history, which is guaranteed by min_backtest_train
    return pd.DataFrame({"date": [next_date], "lag1": [history["notifications"].iloc[-1]],
                          "lag4": [history["notifications"].iloc[-4]], "lag52": [lag52_series.iloc[0]],
                          "week_sin": [np.sin(2 * np.pi * next_week_number / 52)],
                          "week_cos": [np.cos(2 * np.pi * next_week_number / 52)]})


## 13. Forecast Stabilization Helpers

Three small helpers that keep multi-step forecasts from exploding or oscillating.

`_stabilize_forecast()` takes the raw model output and blends it with an anchor value (a weighted mix of recent levels, seasonality, and trend). The blend weight increases with each step so the further out we forecast, the less we trust the raw model and the more we rely on the anchor. The result is then clipped so it can't jump more than `max_step_delta` from the previous value and can't exceed the ceiling.

`_bounded_seasonal_anchor()` computes a seasonal reference from lag52 but caps it relative to recent activity. This prevents a year where the same week had an unusually high spike from pulling the forecast unrealistically high.

`_estimate_recent_trend()` estimates the per-week slope by comparing the mean of the last 4 weeks to the mean of weeks 5-8. If there aren't enough data points it returns zero.

In [ ]:
def _stabilize_forecast(raw_value: float, last_value: float, anchor_value: float, max_step_delta: float, blend_weight: float, ceiling: float) -> float:
    """Blend the raw model prediction with an anchor, then clip so it can't jump
    more than max_step_delta from the previous value or exceed the ceiling."""
    blended = (1.0 - blend_weight) * raw_value + blend_weight * anchor_value
    lower = max(0.0, last_value - max_step_delta)
    upper = min(ceiling, last_value + max_step_delta)
    return float(np.clip(blended, lower, upper))


def _bounded_seasonal_anchor(lag1: float, lag4: float, lag52: float, typical_step_delta: float) -> float:
    """Compute a seasonal reference capped relative to recent activity, so it doesn't
    pull the forecast unrealistically high when last year's same-week was an outlier."""
    recent_anchor = 0.65 * lag1 + 0.35 * lag4
    seasonal_cap = max(recent_anchor * 2.2, recent_anchor + 3.0 * typical_step_delta)
    return float(min(lag52, seasonal_cap))


def _estimate_recent_trend(series: pd.Series) -> float:
    """Estimate the per-week trend by comparing the mean of the last 4 weeks to weeks 5-8."""
    recent_short = series.tail(4)
    recent_prev = series.tail(8).head(4)
    if len(recent_short) == 4 and len(recent_prev) == 4:
        return float((recent_short.mean() - recent_prev.mean()) / 4.0)
    return 0.0

## 14. Multi-Step Display Model Selector

`choose_multi_step_display_model()` picks which learned model's forecast to show on the monitoring chart. Baselines are excluded since they're not designed to extrapolate. For each remaining model, it measures three things about the forecast trajectory: where it ends up relative to the latest actual (final ratio), the biggest single-step jump anywhere in the forecast (jump ratio), and the peak value reached (max ratio).

A penalty is applied when: the final value is more than 2.5x the latest actual, a single step jumps more than 1x the latest actual, or the peak exceeds 3x the latest actual. The penalized score is the model's backtest score (selection score or MAE) multiplied by the penalty, so a model that's accurate in backtesting but produces a wild trajectory still gets downranked. The model with the lowest penalized score wins.

In [ ]:
def choose_multi_step_display_model(ranking_frame: pd.DataFrame, forecast_df: pd.DataFrame,
        latest_actual: float, preferred_model: str) -> str:
    """Pick which model's forecast to show on the monitoring chart for multi-step forecasts.
    Penalizes models whose trajectories look unrealistic (huge spikes, big jumps, wild endpoints)."""
    baseline_models = {"Naive", "Seasonal Naive"}
    # baselines are excluded because they're not meant to extrapolate, we want a learned model here
    candidate_models = [m for m in ranking_frame.get("Model", pd.Series(dtype=object)).tolist()
        if m in forecast_df.columns and m not in baseline_models]
    if not candidate_models:
        return preferred_model

    latest_scale = max(float(latest_actual), 1.0)
    scored_candidates = []
    for model_name in candidate_models:
        series = np.asarray(forecast_df[model_name].to_numpy(dtype=float), dtype=float)
        if len(series) == 0 or not np.isfinite(series).all():
            continue

        # measure how wild the forecast looks relative to the latest actual value
        max_ratio = float(series.max()) / latest_scale    # how high does it peak
        final_ratio = float(series[-1]) / latest_scale   # where does it end up
        jump_ratio = float(np.abs(np.diff(np.concatenate([[latest_actual], series]))).max()) / latest_scale  # biggest single-step jump
        # penalty kicks in when the forecast ends at 2.5x+, jumps 1x+ in a week, or peaks at 3x+
        penalty = (1.0 + max(0.0, final_ratio - 2.5) + 0.5 * max(0.0, jump_ratio - 1.0) + 0.25 * max(0.0, max_ratio - 3.0))

        # use selection score if available, fall back to MAE
        if "Selection Score" in ranking_frame.columns:
            base_series = ranking_frame.loc[ranking_frame["Model"] == model_name, "Selection Score"]
        elif "MAE" in ranking_frame.columns:
            base_series = ranking_frame.loc[ranking_frame["Model"] == model_name, "MAE"]
        else:
            base_series = pd.Series(dtype=float)
        base_value = float(base_series.iloc[0]) if len(base_series) > 0 else float("inf")
        # penalized score = raw accuracy score * trajectory penalty
        scored_candidates.append((base_value * penalty, base_value, model_name))

    if not scored_candidates:
        return preferred_model

    # lowest penalized score wins
    scored_candidates.sort()
    return str(scored_candidates[0][2])


## 15. Multi-Step Forecasting Engine

`build_multi_step_forecast()` generates predictions for multiple weeks ahead using a recursive approach: each step builds features from the most recent known or predicted values, gets a raw prediction, stabilizes it, and feeds the result back as lag1 for the next step. Each model keeps its own rolling history so its predictions only feed back into its own future inputs, not a shared pool.

Three mechanisms keep forecasts from going off the rails. First, anchor blending: the raw model output is mixed with an anchor (recent median level, seasonal signal, and trend estimate), and the anchor weight increases from 30% at step 1 up to 82% at step 6+, so the further out we go, the less we trust the model's extrapolation. Second, delta limiting: the maximum change per step scales with the square root of the step number, so small changes are allowed early but larger ones are permitted further out. Third, a ceiling prevents forecasts from exceeding roughly 1.4x the recent 26-week peak or 3x the latest actual value.

Naive just returns lag1 (flat line) and Seasonal Naive uses the bounded seasonal anchor. Only learned models (Linear Regression, Random Forest, Negative Binomial) go through `_stabilize_forecast()`.

In [ ]:
def build_multi_step_forecast(history: pd.DataFrame, bundle: Dict[str, Dict[str, Any]], horizon: int) -> pd.DataFrame:
    """Generate predictions for multiple weeks into the future using a recursive (autoregressive) approach.
    Each step feeds its prediction back as lag1 for the next step. Learned models get stabilized
    by blending with an anchor (recent levels + seasonality + trend) to prevent runaway forecasts."""
    if horizon < 1:
        raise ValueError("horizon must be at least 1")

    model_names = list(bundle.keys())
    base = history[["date", "notifications"]].copy().reset_index(drop=True)
    # Each model maintains its own rolling series (since each feeds back its own predictions)
    model_series = {name: pd.DataFrame(base.copy()) for name in model_names}
    rows: List[Dict[str, Union[float, int, pd.Timestamp]]] = []

    # Compute the typical week-to-week change magnitude (90th percentile) for delta limiting
    recent_diffs = history["notifications"].diff().dropna().tail(min(26, max(1, len(history) - 1)))
    typical_step_delta = float(np.quantile(np.abs(recent_diffs), 0.9)) if len(recent_diffs) > 0 else 25.0
    typical_step_delta = max(typical_step_delta, 20.0)
    # Build a seasonal profile: median notifications by epi week
    seasonal_profile = history.groupby(history["date"].dt.isocalendar().week.astype(int))["notifications"].median().to_dict()
    # Set a ceiling so forecasts can't go unreasonably high
    hist_max = float(history["notifications"].max())
    recent_peak = float(history["notifications"].tail(26).max()) if len(history) > 0 else hist_max
    ceiling = max(hist_max * 0.9, recent_peak * 1.4, float(history["notifications"].iloc[-1]) * 3.0, 75.0)

    for step in range(1, horizon + 1):
        next_date = history["date"].max() + pd.Timedelta(weeks=step)
        week_number = int(next_date.isocalendar().week)
        row: Dict[str, Union[float, int, pd.Timestamp]] = {"date": next_date, "step": step}
        blend_weight = min(0.82, 0.30 + 0.10 * max(0, step - 1))  # more blending the further out we go

        for model_name in model_names:
            series = model_series[model_name]
            # Build features from this model's rolling history
            lag1 = float(series["notifications"].iloc[-1])
            lag4 = float(series["notifications"].iloc[-4]) if len(series) >= 4 else lag1
            recent_level = float(series["notifications"].tail(8).median())
            trend_per_week = _estimate_recent_trend(series["notifications"])
            lag52_date = next_date - pd.Timedelta(weeks=52)
            lag52_series = series.loc[series["date"] == lag52_date, "notifications"]
            lag52 = float(lag52_series.iloc[0]) if len(lag52_series) > 0 else lag1
            seasonal_median = float(seasonal_profile.get(week_number, lag52))

            # Create feature row and get raw model prediction
            feat = pd.DataFrame({"date": [next_date], "lag1": [lag1], "lag4": [lag4], "lag52": [lag52],
                                  "week_sin": [np.sin(2 * np.pi * week_number / 52)],
                                  "week_cos": [np.cos(2 * np.pi * week_number / 52)]})
            raw_predictions = predict_with_bundle(bundle, feat)
            raw_value = max(0.0, float(raw_predictions[model_name][0]))

            # Build anchor: weighted mix of recent levels, seasonal signal, and trend
            recent_anchor = 0.55 * lag1 + 0.20 * lag4 + 0.25 * recent_level
            seasonal_reference = 0.5 * lag52 + 0.5 * seasonal_median
            seasonal_cap = max(recent_anchor * 1.45, recent_anchor + 1.75 * typical_step_delta)
            seasonal_anchor = float(min(seasonal_reference, seasonal_cap))
            trend_target = max(0.0, recent_anchor + trend_per_week * min(step, 4))
            anchor_value = 0.65 * recent_anchor + 0.20 * seasonal_anchor + 0.15 * trend_target
            # Allow larger jumps as we go further out (sqrt scaling)
            delta_limit = max(typical_step_delta * (0.65 + 0.20 * math.sqrt(step)), 25.0)

            # Naive just predicts last value; Seasonal Naive uses bounded seasonal anchor; others get stabilized
            if model_name == "Naive":
                forecast_value = lag1
            elif model_name == "Seasonal Naive":
                forecast_value = seasonal_anchor
            else:
                forecast_value = _stabilize_forecast(raw_value=raw_value, last_value=lag1,
                    anchor_value=anchor_value, max_step_delta=delta_limit,
                    blend_weight=min(0.92, 0.55 + 0.08 * max(0, step - 1)), ceiling=ceiling)

            row[model_name] = forecast_value
            # Feed this prediction back into the model's series for the next step
            model_series[model_name] = pd.concat([series,
                pd.DataFrame({"date": [next_date], "notifications": [forecast_value]})], ignore_index=True)
        rows.append(row)

    return pd.DataFrame(rows)


## 16. Error Tables & Holdout Management

`build_error_table()` computes MAE (mean absolute error) and RMSE (root mean squared error) for each model on a given backtest slice and returns a DataFrame sorted by MAE. MAE measures the average absolute miss in notification counts. RMSE penalizes large misses more heavily because errors get squared before averaging.

This function gets called three times on different slices: the full backtest, the holdout period, and the pre-holdout selection window. Each slice tells a different story: full backtest shows overall behavior, holdout shows real-world generalization, and the selection window is what actually decides which model goes to production.

`determine_holdout_start()` finds the date that splits the backtest into those periods. It defaults to the last 52 weeks but enforces a minimum of 26 weeks so we always have a meaningful holdout even if the total backtest is short.

In [ ]:
def build_error_table(bt_slice: pd.DataFrame, model_cols: Dict[str, str]) -> pd.DataFrame:
    """Compute MAE and RMSE for each model on a backtest slice, return sorted by accuracy."""
    rows: List[Dict[str, Union[str, float]]] = []
    for model_name, col in model_cols.items():
        sample = bt_slice.dropna(subset=[col])  # skip weeks where this model has no prediction
        if len(sample) == 0:
            continue
        rows.append({"Model": model_name,
                      "MAE": float(mean_absolute_error(sample["actual"], sample[col])),
                      "RMSE": float(np.sqrt(mean_squared_error(sample["actual"], sample[col])))})
    if not rows:
        return pd.DataFrame(columns=["Model", "MAE", "RMSE"])
    # sort by MAE first, RMSE as tiebreaker
    return pd.DataFrame(rows).sort_values(by=["MAE", "RMSE"]).reset_index(drop=True)


def determine_holdout_start(backtest_results: pd.DataFrame, holdout_weeks: int = DEFAULT_HOLDOUT_WEEKS, minimum_holdout_weeks: int = 26) -> pd.Timestamp:
    """Find the date that splits the backtest into training and holdout periods.
    Default: last 52 weeks are holdout, minimum 26 weeks."""
    unique_dates = sorted(pd.unique(backtest_results["date"]))
    if len(unique_dates) == 0:
        raise ValueError("Backtest results are empty; cannot determine holdout start.")
    # clamp so we always have at least 26 holdout weeks even if the backtest is short
    effective_holdout_weeks = min(len(unique_dates), max(minimum_holdout_weeks, holdout_weeks))
    return pd.Timestamp(unique_dates[-effective_holdout_weeks])


## 17. Predictor Diagnostics

`build_predictor_diagnostics()` builds a small summary table for the production model showing which model is in use, when the holdout window starts, the latest tested week's actual vs predicted values, and the recent 8-week MAE. This is what shows up in the Model Notes section of the dashboard.

In [ ]:
def build_predictor_diagnostics(backtest_results: pd.DataFrame, model_cols: Dict[str, str], prod_name: str, holdout_start: pd.Timestamp) -> pd.DataFrame:
    """Build a summary table for the production model: which model, holdout window,
    latest actual vs predicted, and recent 8-week MAE. Shown in dashboard model notes."""
    prod_col = model_cols[prod_name]
    prod_backtest = backtest_results.dropna(subset=[prod_col]).copy().reset_index(drop=True)
    latest_eval = prod_backtest.iloc[-1]  # the most recent week the backtest has a prediction for
    latest_abs_error = abs(float(latest_eval["actual"]) - float(latest_eval[prod_col]))
    # recent 8-week MAE uses only the holdout period so it reflects real out-of-sample performance
    recent_holdout = prod_backtest[prod_backtest["date"] >= holdout_start].tail(8)
    recent_holdout_mae = (float(mean_absolute_error(recent_holdout["actual"], recent_holdout[prod_col]))
        if len(recent_holdout) > 0 else float("nan"))
    return pd.DataFrame({"Metric": ["Production model", "Testing window start", "Latest tested week", "Latest tested actual",
                    "Latest tested prediction", "Latest tested absolute error", "Recent holdout MAE"],
        "Value": [prod_name, str(pd.Timestamp(holdout_start).date()), str(pd.Timestamp(latest_eval["date"]).date()),
                  f"{int(latest_eval['actual']):,}", f"{float(latest_eval[prod_col]):.1f}",
                  f"{latest_abs_error:.1f}", f"{recent_holdout_mae:.1f}" if np.isfinite(recent_holdout_mae) else "N/A"]})


## 18. Composite Model Selection Scoring

`build_selection_score_table()` ranks models using a weighted composite score across five criteria. Lower score is better. The model with the lowest score becomes the production model.

| Weight | Component | Why it's included |
|---|---|---|
| 45% | Recent MAE | Raw accuracy is the most important thing |
| 20% | Latest absolute error | Recency signal: how well it did on the very last week in the window |
| 20% | 1 - Direction accuracy | Penalizes models that consistently get the trend direction wrong |
| 20% | 1 - Rising F1 | Penalizes models that miss rising weeks or cry wolf too often |
| 15% | False alarm rate | Extra penalty for false alarms specifically |

Before combining, MAE and latest error are each divided by their median across all models. This normalization step means a metric with a larger absolute scale (like MAE in the hundreds) doesn't unfairly dominate over a metric between 0 and 1 (like direction accuracy).

In [ ]:
def build_selection_score_table(evaluation_bt: pd.DataFrame, model_cols: Dict[str, str]) -> pd.DataFrame:
    """Compute a composite score to rank models for production use.
    Weighs recent MAE (45%), latest error (20%), direction accuracy (20%),
    rising F1 (20%), and false alarm rate (15%). Lower = better."""
    rows: List[Dict[str, Union[str, float]]] = []
    for model_name, col in model_cols.items():
        sample = evaluation_bt.dropna(subset=[col]).copy()
        if len(sample) == 0:
            continue
        score = score_predictions(sample["actual"].values, sample[col].values, sample["previous_week"].values)
        recent_mae = float(mean_absolute_error(sample["actual"], sample[col]))
        # latest_abs_error is the error on the very last week in the window, a recency signal
        latest_abs_error = abs(float(sample.iloc[-1]["actual"]) - float(sample.iloc[-1][col]))
        rows.append({"Model": model_name, "Recent MAE": recent_mae, "Latest Abs Error": latest_abs_error,
                      "Direction Acc.": float(score["Direction Acc."]), "Rising F1": float(score["Rising F1"]),
                      "False Alarm Rate": float(score["False Alarm Rate"])})

    if not rows:
        return pd.DataFrame(columns=["Model", "Recent MAE", "Latest Abs Error", "Direction Acc.",
                                      "Rising F1", "False Alarm Rate", "Selection Score"])

    score_table = pd.DataFrame(rows)
    # normalize by the median so no single metric dominates just because it's on a larger scale
    mae_scale = max(float(score_table["Recent MAE"].median()), 1.0)
    latest_scale = max(float(score_table["Latest Abs Error"].median()), 1.0)
    # weighted sum: accuracy matters most (45%), recency and direction each add 20%, false alarms 15%
    score_table["Selection Score"] = (0.45 * (score_table["Recent MAE"] / mae_scale)
        + 0.20 * (score_table["Latest Abs Error"] / latest_scale)
        + 0.20 * (1.0 - score_table["Direction Acc."])
        + 0.20 * (1.0 - score_table["Rising F1"])
        + 0.15 * score_table["False Alarm Rate"])
    score_order = list(np.argsort(score_table["Selection Score"].to_numpy(dtype=float)))
    return pd.DataFrame(score_table.iloc[score_order].copy()).reset_index(drop=True)


## 19. Master Orchestrator - build_monitoring_state()

This is the main function the dashboard calls. It chains every step in the pipeline into one call: load data, build features, run the full expanding-window backtest, split into selection and holdout periods, score models on the pre-holdout window, pick the production and display models, generate the forecast, compute safety bands, and derive trend and risk labels. It returns a single dict with everything the dashboard needs so the frontend only has to call this once.

A few design decisions worth knowing about. The selection window is always the last N weeks before the holdout, never inside it, so model scoring never touches the holdout data. The production model and the display model can be different: the production model always drives trend, risk, and safety band calculations, while the display model is chosen separately based on which trajectory looks most realistic in a multi-step chart. Negative Binomial gets a status field (off, active, or omitted) so the dashboard can warn the user if they enabled it but the fit didn't converge.

Safety bands use the 25th to 75th percentile of the production model's residuals on the holdout period, widened by the square root of the forecast step so uncertainty grows appropriately with horizon.

In [ ]:
def build_monitoring_state(horizon: int = 1, include_nb: bool = True, refresh_data: bool = False,
        holdout_weeks: int = DEFAULT_HOLDOUT_WEEKS, selection_weeks: int = DEFAULT_SELECTION_WEEKS,
        uf_code: int = TARGET_UF_CODE) -> Dict[str, Any]:
    """Main entry point for the dashboard. Chains every step into one call:
    load data -> engineer features -> backtest -> score models -> pick production model ->
    forecast -> derive trend/risk/safety bands -> return everything as a single dict."""

    # Load data (uses parquet cache unless refresh_data is True)
    available_data_urls = get_available_data_urls(refresh=refresh_data)
    available_years = sorted(available_data_urls)
    raw_all, full_df, df = build_weekly_series(refresh=refresh_data, uf_code=uf_code)
    features_df = create_features(df)
    backtest_results, model_cols = run_backtest(features_df, include_nb=include_nb)

    # Track whether Negative Binomial was requested, converged, or had to be skipped
    nb_requested = include_nb
    nb_available = "Negative Binomial" in model_cols
    if nb_requested and not nb_available:
        nb_status = "omitted"  # requested but fit did not converge
    elif nb_requested and nb_available:
        nb_status = "active"
    else:
        nb_status = "off"

    # Split backtest into selection (pre-holdout) and holdout periods
    full_err = build_error_table(backtest_results, model_cols)
    holdout_start = determine_holdout_start(backtest_results, holdout_weeks=holdout_weeks)
    selection_bt = backtest_results[backtest_results["date"] < holdout_start].copy()
    holdout_bt = backtest_results[backtest_results["date"] >= holdout_start].copy()
    holdout_err = build_error_table(holdout_bt, model_cols) if len(holdout_bt) > 0 else full_err.copy()

    # Selection window: last N weeks BEFORE the holdout (no leakage into holdout)
    recent_selection_bt = selection_bt.tail(selection_weeks).copy() if len(selection_bt) > 0 else selection_bt.copy()
    recent_err = build_error_table(recent_selection_bt, model_cols) if len(recent_selection_bt) > 0 else holdout_err.copy()
    selection_score_table = build_selection_score_table(recent_selection_bt, model_cols) if len(recent_selection_bt) > 0 else pd.DataFrame()

    # Rank models and pick the production model (best composite score)
    ranking_frame = pd.DataFrame()
    if len(selection_score_table) > 0:
        ranking_frame = selection_score_table[["Model", "Selection Score"]].copy()
        ranking_basis = f"Pre-holdout selection score ({len(recent_selection_bt)} weeks before holdout)"
    elif len(recent_err) > 0:
        ranking_frame = recent_err
        ranking_basis = f"Pre-holdout selection focus ({len(recent_selection_bt)} weeks)"
    elif len(holdout_err) > 0:
        ranking_frame = holdout_err
        ranking_basis = f"Holdout evaluation ({len(holdout_bt)} weeks)"
    else:
        ranking_frame = full_err
        ranking_basis = "Full expanding-window backtest"

    ranking_frame = pd.DataFrame(ranking_frame).reset_index(drop=True)
    prod_name = ranking_frame.iloc[0]["Model"] if len(ranking_frame) > 0 else "Naive"
    prod_col = model_cols[prod_name]

    # Find the best learned (non-baseline) model as a benchmark
    learned_models = [m for m in model_cols if m not in {"Naive", "Seasonal Naive"}]
    # best learned model is used as a benchmark even when a baseline wins
    learned_ranking_source = recent_err if len(recent_err) > 0 else holdout_err if len(holdout_err) > 0 else full_err
    prod_learned_ranking = learned_ranking_source[learned_ranking_source["Model"].isin(learned_models)].copy()
    if len(prod_learned_ranking) > 0:
        learned_order = np.argsort(prod_learned_ranking["MAE"].to_numpy(dtype=float))
        prod_learned_ranking = prod_learned_ranking.iloc[learned_order].reset_index(drop=True)
    if prod_name in learned_models and len(prod_learned_ranking) > 1:
        prod_learned_name = str(prod_learned_ranking.iloc[1]["Model"])
    elif prod_name not in learned_models and len(prod_learned_ranking) > 0:
        prod_learned_name = str(prod_learned_ranking.iloc[0]["Model"])
    else:
        prod_learned_name = prod_name

    # Generate the forecast (single-step or multi-step)
    prod_bundle = fit_model_bundle(features_df, include_nb=include_nb, verbose=False)
    if horizon <= 1:
        next_features = build_next_week_features(df)
        next_predictions = predict_with_bundle(prod_bundle, next_features)
        next_date = pd.Timestamp(next_features["date"].iloc[0])
        forecast_df = pd.DataFrame({"date": [next_date], "step": [1],
            **{name: [float(values[0])] for name, values in next_predictions.items()}})
    else:
        forecast_df = build_multi_step_forecast(df, prod_bundle, horizon)
        next_date = pd.Timestamp(forecast_df["date"].iloc[0])

    # Pick display model for multi-step charts (may differ from production model for visual stability)
    if horizon > 1:
        display_model = choose_multi_step_display_model(ranking_frame, forecast_df,
            latest_actual=float(df["notifications"].iloc[-1]),
            preferred_model=prod_learned_name if prod_name in {"Naive", "Seasonal Naive"} else prod_name)
    else:
        display_model = prod_name

    # Safety bands from production model residual quantiles, widening with sqrt(step)
    residual_source = holdout_bt if len(holdout_bt) >= 8 else backtest_results
    residuals = residual_source["actual"].values - residual_source[prod_col].values
    valid_residuals = residuals[np.isfinite(residuals)]
    r_lo, r_hi = np.quantile(valid_residuals, BAND_QUANTILES)
    steps = forecast_df["step"].values
    forecast_df["lower"] = np.clip(forecast_df[display_model].values + r_lo * np.sqrt(steps), 0, None)
    forecast_df["upper"] = np.clip(forecast_df[display_model].values + r_hi * np.sqrt(steps), 0, None)

    # Headline values for the dashboard cards
    latest_actual = int(df["notifications"].iloc[-1])
    latest_date = pd.Timestamp(df["date"].iloc[-1])
    prod_forecast = float(forecast_df[prod_name].iloc[0]) if prod_name in forecast_df.columns else float(forecast_df[display_model].iloc[0])  # for trend/risk
    display_forecast = float(forecast_df[display_model].iloc[0])  # for chart headline
    learned_forecast = float(forecast_df[prod_learned_name].iloc[0])
    next_lower, next_upper = float(forecast_df["lower"].iloc[0]), float(forecast_df["upper"].iloc[0])
    final_forecast = float(forecast_df[display_model].iloc[-1])

    # Trend label: compare production forecast to latest actual
    pct_change = (prod_forecast - latest_actual) / max(latest_actual, 1)
    if pct_change > TREND_THRESHOLD:
        trend_label, trend_icon = "Rising", "\U0001f4c8"
    elif pct_change < -TREND_THRESHOLD:
        trend_label, trend_icon = "Falling", "\U0001f4c9"
    else:
        trend_label, trend_icon = "Stable", "\u27a1\ufe0f"

    # Risk label: compare forecast to historical percentiles
    q50, q75, q90 = df["notifications"].quantile([0.50, 0.75, 0.90]).values
    if prod_forecast >= q90:
        risk_label, risk_icon = "Very High", "\U0001f534"
    elif prod_forecast >= q75:
        risk_label, risk_icon = "High", "\U0001f7e0"
    elif prod_forecast >= q50:
        risk_label, risk_icon = "Moderate", "\U0001f7e1"
    else:
        risk_label, risk_icon = "Low", "\U0001f7e2"

    # Coverage: what share of holdout weeks fell inside the safety band?
    fr_card = holdout_bt.copy()
    fr_card["pl"] = np.clip(fr_card[prod_col] + r_lo, 0, None)
    fr_card["pu"] = np.clip(fr_card[prod_col] + r_hi, 0, None)
    cov = ((fr_card["actual"] >= fr_card["pl"]) & (fr_card["actual"] <= fr_card["pu"])).mean() if len(fr_card) > 0 else np.nan
    recent_8 = holdout_bt.tail(8)
    recent_mae = mean_absolute_error(recent_8["actual"], recent_8[prod_col]) if len(recent_8) > 0 else np.nan
    predictor_diagnostics = build_predictor_diagnostics(backtest_results, model_cols, prod_name, holdout_start)

    # Cache metadata for dashboard attribution
    _cache_info = get_cache_info(uf_code=uf_code)
    uf_label = UF_OPTIONS.get(uf_code, f"UF {uf_code}")

    # Return everything the dashboard needs in a single dict
    # pack everything into one dict so dashboard.py only needs a single call
    return {"available_data_urls": available_data_urls, "available_years": available_years,
        "raw_all": raw_all, "full_df": full_df, "df": df,
        "features_df": features_df, "backtest_results": backtest_results,
        "model_cols": model_cols, "full_err": full_err,
        "holdout_start": holdout_start, "selection_bt": selection_bt,
        "holdout_bt": holdout_bt, "holdout_err": holdout_err,
        "recent_selection_bt": recent_selection_bt, "recent_err": recent_err,
        "selection_score_table": selection_score_table, "ranking_frame": ranking_frame,
        "ranking_basis": ranking_basis, "prod_name": prod_name, "prod_col": prod_col,
        "prod_learned_name": prod_learned_name, "prod_learned_ranking": prod_learned_ranking,
        "prod_bundle": prod_bundle, "forecast_df": forecast_df,
        "next_date": next_date, "display_model": display_model,
        "r_lo": float(r_lo), "r_hi": float(r_hi),
        "latest_actual": latest_actual, "latest_date": latest_date,
        "prod_forecast": prod_forecast, "display_forecast": display_forecast,
        "learned_forecast": learned_forecast, "next_lower": next_lower,
        "next_upper": next_upper, "final_forecast": final_forecast,
        "pct_change": float(pct_change), "trend_label": trend_label,
        "trend_icon": trend_icon, "risk_label": risk_label, "risk_icon": risk_icon,
        "cov": float(cov) if np.isfinite(cov) else np.nan,
        "recent_mae": float(recent_mae) if np.isfinite(recent_mae) else np.nan,
        "predictor_diagnostics": predictor_diagnostics, "nb_status": nb_status,
        "uf_code": uf_code, "uf_label": uf_label,
        "data_source": DATA_SOURCE_LABEL, "cache_info": _cache_info}


---

## Pipeline

```
SINAN CSVs (S3)  →  load_dengue_year()  →  build_weekly_series()
                                                    ↓
                                           create_features()
                                                    ↓
                                    ┌───── run_backtest() ─────┐
                                    │  (expanding window)      │
                                    │  fit → predict → record  │
                                    └──────────────────────────┘
                                                    ↓
                              build_error_table() + build_selection_score_table()
                                                    ↓
                                      Select production model
                                                    ↓
                              fit_model_bundle() on full training data
                                                    ↓
                          build_multi_step_forecast() or single-step
                                                    ↓
                              Safety bands from residual quantiles
                                                    ↓
                             Trend + Risk labels → Dashboard state
```